# UAVCAN Intrusion Dataset Preprocessing
**Source:** Korea University HCRL Lab (IEEE DataPort)  
**Attack Types:** Flooding, Fuzzy, Replay (10 scenarios)  
**Binary Format:** Label, Timestamp, Interface, CAN_ID, DLC, Data  
**Output:** processed_UAVCAN.csv

## Dataset Domain Knowledge — UAVCAN Drone Communication
### How Drones Communicate Internally
Every component inside a drone (motor, GPS, battery, sensors) 
communicates through the **CAN bus** — a shared internal highway. 
Each component sends messages that other components read and act on.

### What Each Column Means

| Column | Meaning | Attack Signal |
|--------|---------|---------------|
| `CAN_ID`      | Who is sending the message — each component has a unique ID  | Unknown IDs = possible attacker |
| `DLC`         | Data Length Code — how many bytes in the message (0–8)       | Wrong length = Fuzzy attack |
| `byte0–byte7` | The actual message content (motor speed, GPS, battery, etc.) | Random bytes = Fuzzy; repeated bytes = Replay |

### The 3 Attack Types

**Flooding** — spam attack
- Attacker sends thousands of fake messages per second
- CAN bus gets jammed, real messages can't get through
- Drone loses control

**Fuzzy** — confusion attack  
- Attacker sends random garbage data in the byte fields
- Drone components receive nonsense commands
- Drone behaves unpredictably or crashes

**Replay** — copy-paste attack
- Attacker records old legitimate messages and resends them
- Drone thinks it received a real command at the wrong time
- Drone performs unintended actions

### Expected XAI Findings
- **SHAP on CAN_ID** → key for detecting Flooding (unknown IDs appear)
- **SHAP on DLC** → key for detecting Fuzzy (wrong message lengths)
- **SHAP on byte patterns** → key for detecting Replay (repeated identical bytes)
- **LIME** → local explanation per CAN message, may highlight different byte patterns than SHAP
- **Permutation Importance** → confirms which features cause biggest F1 drop when shuffled
- **Key question** → do SHAP, LIME, and PI agree on CAN_ID vs DLC vs byte features?

This domain knowledge will help interpret SHAP, LIME, Permutation Importance results later.


In [ ]:
# Binary format source: Kim et al. (2022) "UAVCAN Dataset Description"
# Korea University HCRL Lab, arXiv:2212.09268
# Format: Label, Timestamp, Interface, CAN_ID, DLC, Data
# Each line follows SocketCAN output format

In [2]:
import pandas as pd          # for creating and saving the final CSV table
import numpy as np           # for numerical operations
import zipfile               # for reading .bin files directly from inside the zip
import re                    # for parsing each line of the binary file using regex
from sklearn.preprocessing import LabelEncoder, StandardScaler  # for encoding labels and scaling features

In [4]:
# cell 3:
# what: open 10 .bin filres, then reads them line by line, pull out useful information from eac line, and builds one big table
# why: pandas directly not able to read .bin file, it is raw binary text, we need to manually extract each field from every line before we can build table the model can use.
#.bin file = a messy handwritten note
# Cell 3   = a person who reads it and 
#          types it into a clean spreadsheet

# how it it work steps:
# Step 1: Open the zip file
#         → no need to extract to disk first
# Step 2: Loop through all 10 .bin files
# Step 3: Read each line — example:
#        "Normal (000.000000) can0 10015501 [8] 00 00 00 C0..."
# Step 4: Use regex (pattern) to split the line into pieces:
#        Label     = "Normal"
#        Timestamp = 000.000000
#        CAN_ID    = 10015501 (hex) → converted to integer
#        DLC       = 8 (how many bytes of data)
#       Data      = 00 00 00 C0... (up to 8 bytes)
# Step 5: If data has less than 8 bytes → fill missing with 0
#        (like filling empty cells in Excel)
# Step 6: Add that row to our list
# Step 7: After all files done → convert list to one big table

# zip file path
zip_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAVCAN/UAVCAN_Attack_dataset(DroneCAN).zip"

# regex pattern to extract each field from one line
# Example line: Normal (000.000000)  can0  10015501   [8]  00 00 00 00 08 00 00 C0
# Binary format source: Kim et al. (2022) "UAVCAN Dataset Description" arXiv:2212.09268
pattern = re.compile(
    r'(\w+)\s+\((\d+\.\d+)\)\s+(\w+)\s+([0-9A-Fa-f]+)\s+\[(\d+)\]\s+(.*)'
)

all_rows = []  # collect all parsed rows before building dataframe

with zipfile.ZipFile(zip_path, 'r') as z:
    # loop through all 10 .bin files
    for filename in sorted(z.namelist()):
        if filename.endswith('.bin'):
            print(f"Parsing: {filename}")
            with z.open(filename) as f:
                for line in f:
                    # decode binary line to text
                    line = line.decode('utf-8', errors='ignore').strip()
                    match = pattern.match(line)
                    if match:
                        label     = match.group(1)   # Normal or Attack
                        timestamp = float(match.group(2))
                        interface = match.group(3)   # can0
                        can_id    = int(match.group(4), 16)  # hex to integer
                        dlc       = int(match.group(5))      # data length
                        data_bytes = match.group(6).split()  # payload bytes

                        # pad payload to 8 bytes with 0 if shorter
                        data_bytes = [int(b, 16) for b in data_bytes]
                        data_bytes += [0] * (8 - len(data_bytes))

                        all_rows.append([label, timestamp, can_id, dlc] + data_bytes)

# build dataframe with column names
columns = ['label', 'timestamp', 'CAN_ID', 'DLC',
           'byte0', 'byte1', 'byte2', 'byte3',
           'byte4', 'byte5', 'byte6', 'byte7']
df = pd.DataFrame(all_rows, columns=columns)

print(f"\nTotal rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"\nLabel counts:\n{df['label'].value_counts()}")

Parsing: type10_label.bin
Parsing: type1_label.bin
Parsing: type2_label.bin
Parsing: type3_label.bin
Parsing: type4_label.bin
Parsing: type5_label.bin
Parsing: type6_label.bin
Parsing: type7_label.bin
Parsing: type8_label.bin
Parsing: type9_label.bin

Total rows: 2032530
Total columns: 12

Label counts:
label
Normal    1276014
Attack     756516
Name: count, dtype: int64


In [5]:
# ============================================================
# CELL 4: DROP UNNECESSARY COLUMNS + CHECK DATA QUALITY
# ============================================================
# WHAT: Remove timestamp and interface columns, then check
#       for missing (NaN) and infinite (Inf) values
#
# WHY:  timestamp = time value, not a behavioral feature.
#       Keeping it would cause data leakage — model memorizes
#       WHEN attacks happen instead of HOW they look.
#       interface = always 'can0' for every single row.
#       A column with only one value adds zero information
#       to the model — like a column where every student
#       has the same grade. It teaches nothing.
#
# HOW:  Step 1: Drop timestamp and interface columns
#       Step 2: Count total NaN (empty cells)
#       Step 3: Count total Inf (infinity values)
#       Step 4: Print shape to confirm columns removed
# ============================================================

# drop timestamp and interface — neither is a useful feature
# 'interface' => Normal (000.000000)  can0  10015501   [8]  00 00 00 00 08 00 00 C0
# can0 is name of CAN bus divice on the drone, Every single row in all 10 files has exactly can0 — never changes.
# 'ignore' => "If the column doesn't exist — just skip it quietly, don't crash"
df = df.drop(columns=['timestamp', 'interface'], errors='ignore')

# check for missing values (NaN) — empty cells break model training
# nan = Not a Number - empty/missing value in a cell.it will break model training because the model can not do math on nothing
# df.isnnull() -> checks every single cell, returns True if empty, false if has value
# .sum() -> counts how many true(empty) per column. .sum() -> sum again for totoal number of true
nan_count = df.isnull().sum().sum()
print(f"Total NaN values: {nan_count}")

# check for infinite values (Inf) — division by zero results break training
# df.select_dtypes(include=np.number) → selects only number columns (ignores text like label)
# np.isinf(...) → checks every number cell, returns True if it's infinity
# .sum().sum() → counts total infinity values across all columns
inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
print(f"Total Inf values: {inf_count}")

# confirm shape after dropping columns
print(f"\nShape after dropping columns: {df.shape}")

Total NaN values: 0
Total Inf values: 0

Shape after dropping columns: (2032530, 11)


In [6]:
# ============================================================
# CELL 5: ENCODE LABELS + SCALE FEATURES
# ============================================================
# WHAT: Convert text labels to numbers, then scale all
#       feature columns to the same range
#
# WHY:  Models only understand numbers, not text like
#       "Normal" or "Attack". LabelEncoder translates them.
#       CAN_ID values can be very large (hex converted to int)
#       while DLC is only 0-8 and bytes are 0-255.
#       StandardScaler makes all columns fair and equal
#       so large CAN_ID values don't dominate the model.
#
# HOW:  Step 1: Split table into X (features) and y (labels)
#       Step 2: LabelEncoder converts Normal/Attack → 0/1
#       Step 3: StandardScaler rescales all feature columns
# ============================================================

# separate features and label
# X = everything except label column (input to model)
# y = label column only (what model learns to predict)
X = df.drop(columns=['label'])
y = df['label']

# encode text labels to numbers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# print mapping so we know which number = which label
print("Label encoding:")
for i, label in enumerate(le.classes_):
    print(f"  {label} → {i}")

# scale all features to same range
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nFeatures shape: {X_scaled.shape}")
print(f"Labels shape: {y_encoded.shape}")

Label encoding:
  Attack → 0
  Normal → 1

Features shape: (2032530, 10)
Labels shape: (2032530,)


In [7]:
# ============================================================
# CELL 6: SAVE PROCESSED FILE
# ============================================================
# WHAT: Combine scaled features and encoded labels into one
#       dataframe and save as processed_UAVCAN.csv
#
# WHY:  Model notebooks (RF, CNN, Autoencoder) need one single
#       clean CSV file to load
#
# HOW:  Step 1: Convert X_scaled numpy array back to dataframe
#       Step 2: Add encoded label column
#       Step 3: Save to disk as CSV
# ============================================================

# convert scaled features back to dataframe with original column names
df_processed = pd.DataFrame(X_scaled, columns=X.columns)

# add encoded label column
df_processed['label'] = y_encoded

# save to disk
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAVCAN/processed_UAVCAN.csv"
df_processed.to_csv(save_path, index=False)

print(f"Saved: {save_path}")
print(f"Final shape: {df_processed.shape}")
print(f"\nLabel distribution:")
print(df_processed['label'].value_counts())

Saved: /Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAVCAN/processed_UAVCAN.csv
Final shape: (2032530, 11)

Label distribution:
label
1    1276014
0     756516
Name: count, dtype: int64


In [8]:
# Split train/test 70/30
# stratify = keeps same class ratio in both train and test sets
# random_state=42 = reproducible split every run
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (1422771, 10)
X_test shape: (609759, 10)
y_train shape: (1422771,)
y_test shape: (609759,)


In [10]:
import os
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAVCAN/processed/"
os.makedirs(save_path, exist_ok=True)

np.save(save_path + "X_train.npy", X_train)
np.save(save_path + "X_test.npy", X_test)
pd.DataFrame(y_train).to_csv(save_path + "y_train.csv", index=False)
pd.DataFrame(y_test).to_csv(save_path + "y_test.csv", index=False)
pd.DataFrame(X.columns).to_csv(save_path + "feature_names.csv", index=False)
pd.DataFrame(le.classes_).to_csv(save_path + "label_classes.csv", index=False)

print("Saved!")
print(f"X_train.npy: {X_train.shape}")
print(f"X_test.npy: {X_test.shape}")

Saved!
X_train.npy: (1422771, 10)
X_test.npy: (609759, 10)


## Preprocessing Complete

| Item | Value |
|------|-------|
| Source | Korea University HCRL Lab — UAVCAN Intrusion Dataset |
| Binary format | Label, Timestamp, Interface, CAN_ID, DLC, Data |
| Format reference | Kim et al. (2022) arXiv:2212.09268 |
| Total rows | 2,032,530 |
| Total features | 10 |
| Labels | Normal=1, Attack=0 |
| Class balance | Normal 62.8% / Attack 37.2% |
| Train set | 1,422,771 rows |
| Test set | 609,759 rows |
| Split ratio | 70% train / 30% test |
| Output files | X_train.npy, X_test.npy, y_train.csv, y_test.csv |